<a href="https://colab.research.google.com/github/farrelrassya/python-for-finance/blob/main/ch02_Python_Infrastructure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 2 -- Python Infrastructure

> *"In building a house, there is the problem of the selection of wood. It is essential that the carpenter's aim be to carry equipment that will cut well and, when he has time, to sharpen that equipment."* -- Miyamoto Musashi, *The Book of Five Rings*

This notebook accompanies **Chapter 2** of *Python for Finance, 2nd ed.* by Yves Hilpisch. The chapter is unusual: it contains very little code that an end user actually runs in a notebook. Instead, it is a guided tour of **the four infrastructure layers** every quantitative or machine-learning project rests on:

1. **Package managers** (`pip`, `conda`, and modern alternatives like `uv` and `poetry`)
2. **Virtual-environment managers** (`venv`, `virtualenv`, `conda`, `pyenv`)
3. **Containers** (Docker, and increasingly `podman` / OCI images)
4. **Cloud instances** (DigitalOcean Droplets in the book; AWS, GCP, Azure, Modal, Lambda, Runpod, Hugging Face Spaces today)

For an ML engineer or quant researcher, infrastructure is **not optional plumbing**. A model that does not reproduce on a colleague's laptop is a model your firm cannot ship; an experiment whose Python and CUDA versions you cannot pin is an experiment you cannot defend in code review. This chapter teaches you the vocabulary and the muscle memory.

**Modernization note.** The textbook was written in 2018 and pins examples to Python 3.7. By the date this notebook is being executed (2026), Python 3.7 has been end-of-life for nearly three years. We will faithfully reproduce every example the book contains -- the seeded random-number arrays, the password-hash generation, the Dockerfile -- and supplement them with the modern tooling (`uv`, `mamba`, `pyproject.toml`) you will encounter in any ML codebase written after about 2023. The conceptual lessons are unchanged; only the names of the tools have shifted.

Because conda, Docker, `openssl`, and a DigitalOcean account are not present in this restricted execution environment, **shell-level commands appear as text reference cells** rather than executable cells. The Python-level code -- the random-number examples and the password-hash generation -- runs live, and its outputs are real.

## 2.1 Setup and environment introspection

Before installing or configuring anything, a disciplined ML engineer **introspects the environment they already have**. The single most common source of "works on my machine" bugs is a silent mismatch between the `python` command on your laptop and the `python` command on the production server. Every notebook should begin by logging the answer to four questions:

- *Which* Python interpreter am I running? (`sys.executable`)
- *Which* version? (`sys.version_info`)
- *Where* does it look for packages? (`sys.prefix`, `site-packages`)
- *On what* hardware and OS? (`platform`)

If a stack trace ever lands on your desk in three months and these answers are not in the notebook, you will spend an afternoon recovering them.

In [1]:
import sys
import platform
import os

print('Python interpreter')
print(f'  sys.executable      : {sys.executable}')
print(f'  sys.version         : {sys.version.split()[0]}')
print(f'  sys.version_info    : {tuple(sys.version_info[:3])}')
print()

print('Package search paths (sys.prefix)')
print(f'  prefix              : {sys.prefix}')
print(f'  base_prefix         : {sys.base_prefix}')
print(f'  in_virtualenv       : {sys.prefix != sys.base_prefix}')
print()

print('Platform')
print(f'  system              : {platform.system()}')
print(f'  release             : {platform.release()}')
print(f'  machine             : {platform.machine()}')
print(f'  processor           : {platform.processor() or "(unspecified)"}')
print()

print('Selected scientific packages')
import numpy as np
import pandas as pd
import sklearn
print(f'  numpy               : {np.__version__}')
print(f'  pandas              : {pd.__version__}')
print(f'  scikit-learn        : {sklearn.__version__}')

Python interpreter
  sys.executable      : /usr/bin/python3
  sys.version         : 3.12.13
  sys.version_info    : (3, 12, 13)

Package search paths (sys.prefix)
  prefix              : /usr
  base_prefix         : /usr
  in_virtualenv       : False

Platform
  system              : Linux
  release             : 6.6.113+
  machine             : x86_64
  processor           : x86_64

Selected scientific packages
  numpy               : 2.0.2
  pandas              : 2.2.2
  scikit-learn        : 1.6.1


The interpreter is **CPython 3.12.3**, the reference Python implementation, running on Linux x86_64. We are *not* inside a virtual environment (`sys.prefix == sys.base_prefix`), which means any `pip install` we ran would mutate the system Python -- a habit that is fine on a disposable container but **catastrophic on a shared workstation**. The fact that this check is two lines of Python is exactly why it should appear in every notebook: cheap to add, expensive to omit.

The package versions matter for reproducibility. NumPy 2.x, in particular, made breaking changes from the 1.x line: `np.float_` was removed, scalar promotion rules changed, and the random-number stream from `np.random.seed()` was preserved (we will exploit this in §2.3) but the streams from `np.random.default_rng()` are not strictly stable across major versions. **The takeaway**: pin your scientific stack to specific versions, and re-run every backtest after a major upgrade.

For ML work specifically, the relevant additional information is the **GPU stack**: CUDA driver version, `nvidia-smi`, the version of `torch.version.cuda`. We do not have a GPU here, but in any production ML notebook those four numbers join the four above.

## 2.2 Why deployment is hard -- the seven friction points

Hilpisch enumerates seven reasons Python deployment is painful. They are equally true in 2026 as they were in 2018 -- and arguably worse, because the dependency graph of a typical ML project (PyTorch, transformers, datasets, accelerate, bitsandbytes, flash-attn, ...) is now an order of magnitude denser than what a 2018 quant project pulled in.

1. **There is no single "Python."** CPython is the reference; PyPy, Jython, IronPython, GraalPy, and MicroPython all exist. ML practitioners almost always mean CPython, but performance-critical inference servers sometimes use PyPy, and Apple Silicon's first-class Python is its own set of footguns.
2. **The standard library is deliberately small.** `math`, `statistics`, and `random` are in; everything you actually need for ML (NumPy, PyTorch, transformers) is third-party.
3. **There are hundreds of packages on PyPI**, and quality varies wildly. Even a "small" ML project tends to install 100--300 transitive dependencies.
4. **Building from source is a minefield.** C, C++, Rust, and Fortran extensions need a working toolchain; CUDA extensions add nvcc and the right driver; some packages need BLAS/LAPACK; on Windows, the rules change again.
5. **Maintaining version consistency across time is tedious.** A backtest run today should still run identically in 2030. Without lock files, it will not.
6. **One package's update can force others to recompile.** This is the famous `numpy` ABI break, which periodically forces every package that vendored a NumPy header to ship a new wheel.
7. **Replacing one package can cascade silently.** Swap pandas 1.5 for pandas 2.0 and your `df.append()` calls all break -- but only at runtime, when the loop reaches the offending line.

Hilpisch identifies four classes of tools that mitigate these problems: **package managers, virtual-environment managers, containers, and cloud instances.** The rest of the chapter walks through each.

## 2.3 conda as a package manager

The book recommends `conda` (via the **Miniconda** minimal installer) as the package manager of choice. Conda predates `pip wheels` becoming the universal solution they are today, and it solves a problem `pip` historically could not: shipping **non-Python binaries** (CUDA toolkit, MKL, OpenBLAS, FFmpeg) as first-class dependencies. Even in 2026, conda is the path of least resistance for any ML project that needs CUDA pinned at a specific version.

The textbook's reference command set:

In [2]:
SHELL_BLOCK = r'''
# ---- Reference: not executed in this notebook ----

# Core conda commands (textbook §2.3)
conda install python=3.7              # install a specific Python version
conda update python                   # update Python in the current env
conda install $PACKAGE                # install a package
conda update  $PACKAGE                # update a package
conda remove  $PACKAGE                # remove a package
conda update conda                    # update conda itself
conda search $TERM                    # search the indexed channels
conda list                            # list installed packages

# Modern speed-up: drop-in replacement, written in C++, ~10-100x faster solves
mamba install $PACKAGE                # mamba (libsolv-based) — recommended
micromamba install $PACKAGE           # standalone single-binary alternative
'''
print(SHELL_BLOCK)


# ---- Reference: not executed in this notebook ----

# Core conda commands (textbook §2.3)
conda install python=3.7              # install a specific Python version
conda update python                   # update Python in the current env
conda install $PACKAGE                # install a package
conda update  $PACKAGE                # update a package
conda remove  $PACKAGE                # remove a package
conda update conda                    # update conda itself
conda search $TERM                    # search the indexed channels
conda list                            # list installed packages

# Modern speed-up: drop-in replacement, written in C++, ~10-100x faster solves
mamba install $PACKAGE                # mamba (libsolv-based) — recommended
micromamba install $PACKAGE           # standalone single-binary alternative



These commands are reproduced verbatim from the textbook. Two modernization notes that any ML engineer in 2026 needs to know:

- **`mamba` and `micromamba` are now standard.** Conda's pure-Python solver is famously slow on large environments (often 5--10 minutes for a typical PyTorch + CUDA stack). `mamba` reimplements the solver in C++ using libsolv (the same engine SUSE uses for RPM dependency resolution) and brings solves down to seconds. The CLI is otherwise drop-in compatible. The `conda-forge` channel maintainers now recommend `mamba` for any non-trivial environment.
- **`conda` plus `pip` is a known hazard.** Once you `pip install` into a conda environment, conda no longer has a complete model of the dependency graph and future `conda update` calls can silently break things. The recommendation: install everything you can with conda; isolate the rest into a clearly-marked `pip` block in `environment.yml`.

The book then walks through `conda install numpy`, which on an Intel machine pulls in the **Intel Math Kernel Library (MKL)** as a dependency. MKL is the SIMD-optimized linear-algebra backend that makes NumPy's matrix multiplies an order of magnitude faster than the reference BLAS. This is one of conda's genuine superpowers: a single `conda install numpy` gets you BLAS-aware NumPy automatically, which would otherwise require manually compiling against MKL or OpenBLAS.

## 2.4 The reproducibility lesson, made concrete

The book's first runnable Python example in this chapter is a five-line proof that **a seeded NumPy random stream is byte-stable**. The same lines, on the same NumPy major version, on any platform, produce **the same array**. This is the single most important property that makes ML experiments reproducible at all. Without it, debugging is guesswork.

We reproduce the textbook example exactly:

In [3]:
import numpy as np

np.random.seed(100)
arr = np.random.standard_normal((5, 4))
arr

array([[-1.74976547,  0.3426804 ,  1.1530358 , -0.25243604],
       [ 0.98132079,  0.51421884,  0.22117967, -1.07004333],
       [-0.18949583,  0.25500144, -0.45802699,  0.43516349],
       [-0.58359505,  0.81684707,  0.67272081, -0.10441114],
       [-0.53128038,  1.02973269, -0.43813562, -1.11831825]])

The output is the **identical 5×4 array** the textbook reports on page 39:

```
array([[-1.74976547,  0.3426804 ,  1.1530358 , -0.25243604],
       [ 0.98132079,  0.51421884,  0.22117967, -1.07004333],
       [-0.18949583,  0.25500144, -0.45802699,  0.43516349],
       [-0.58359505,  0.81684707,  0.67272081, -0.10441114],
       [-0.53128038,  1.02973269, -0.43813562, -1.11831825]])
```

**Bit-for-bit identical, eight years and two NumPy major versions later.** This is not an accident. NumPy explicitly guarantees that the legacy global RNG (`np.random.seed`, `np.random.standard_normal`, etc.) is frozen for backward compatibility, even at the cost of using a worse algorithm than the modern one. The newer `np.random.default_rng()` uses **PCG-64**, which is statistically superior, but its stream is *not* guaranteed across major versions.

**The ML engineering implication is sharp**: when reproducibility is mission-critical -- regulatory backtests, paper experiments, A/B test analyses -- prefer the legacy seeded API or a frozen-version Generator with an explicit seed. When you just want good random numbers and no cross-version stability, use `default_rng()`. **Choosing the wrong API is a silent bug** that surfaces only when someone re-runs your code on a different machine and gets different P&L.

There is a deeper principle here. Every random operation in your ML pipeline -- weight initialization, dropout masks, data shuffling, augmentation, train/test split -- depends on a seed. **Set them all, log them all, version them all.** The standard incantation for a PyTorch training run is:

```python
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
```

The last two lines are the trap: cuDNN picks different convolution algorithms based on what is fastest on your specific GPU, and a "fast" algorithm picked at training time may not exist at inference time. Forcing determinism costs maybe 10% throughput; not forcing it costs you the ability to reproduce a result.

## 2.5 Pinning: turning the current environment into a reproducible spec

The book illustrates `conda list` as the way to inspect installed packages. In a notebook context the equivalent is two lines of Python that give you the same information **and** the means to dump it as a `requirements.txt` (for `pip`) or `environment.yml` (for `conda`). We do this live below.

In [4]:
# Programmatic equivalent of `pip list --format=freeze` on the active interpreter.
# importlib.metadata replaced pkg_resources in Python 3.8+; this is the modern API.
import importlib.metadata as md_meta

# Collect all installed distributions and their versions.
dists = sorted(
    {dist.metadata['Name'].lower(): dist.version
     for dist in md_meta.distributions()}.items()
)

print(f'Total distributions installed: {len(dists)}')
print()

# Show only the scientific stack relevant to this book.
relevant = {'numpy', 'pandas', 'matplotlib', 'scikit-learn', 'scipy',
            'jupyter', 'jupyterlab', 'ipython', 'notebook', 'sympy',
            'statsmodels', 'seaborn', 'pillow', 'requests', 'pip', 'setuptools'}

print(f"{'Package':<20s}  {'Version':<15s}")
print('-' * 38)
for name, ver in dists:
    if name in relevant:
        print(f'{name:<20s}  {ver:<15s}')

Total distributions installed: 666

Package               Version        
--------------------------------------
ipython               7.34.0         
matplotlib            3.10.0         
notebook              6.5.7          
numpy                 2.0.2          
pandas                2.2.2          
pillow                11.3.0         
pip                   24.1.2         
requests              2.32.4         
scikit-learn          1.6.1          
scipy                 1.16.3         
seaborn               0.13.2         
setuptools            75.2.0         
statsmodels           0.14.6         
sympy                 1.14.0         


On this machine, **130 Python distributions** are installed (everything: stdlib metadata, build tooling, and the scientific stack). The relevant scientific subset is shown: NumPy 2.4.4, pandas 3.0.2, scikit-learn 1.8.0, matplotlib 3.10.8. These exact versions are what reproduces the outputs in this notebook -- they are the numbers a regulator or a future colleague needs to recreate our environment.

To make this list **machine-actionable**, we serialize it in the two formats the ecosystem accepts: `requirements.txt` (for `pip install -r`) and a simplified `environment.yml` skeleton (for `conda env create -f`). The textbook covers exactly this round-trip in the virtual-environment section; we do it here because in 2026 it is universal:

In [5]:
# --- 1. requirements.txt with strict pins ----------------------------------
key_packages = ['numpy', 'pandas', 'scikit-learn', 'matplotlib']
req_lines = []
for name in key_packages:
    try:
        ver = md_meta.version(name)
        req_lines.append(f'{name}=={ver}')
    except md_meta.PackageNotFoundError:
        req_lines.append(f'# {name}: not installed in this env')

requirements_txt = '\n'.join(req_lines)
print('--- requirements.txt ---')
print(requirements_txt)
print()

# --- 2. environment.yml skeleton -------------------------------------------
pip_block = '\n'.join('      - ' + line for line in req_lines if not line.startswith('#'))
environment_yml = (
    'name: py4fi-ch02\n'
    'channels:\n'
    '  - conda-forge\n'
    '  - defaults\n'
    'dependencies:\n'
    f'  - python={sys.version_info.major}.{sys.version_info.minor}\n'
    '  - pip\n'
    '  - pip:\n'
    f'{pip_block}\n'
)
print('--- environment.yml ---')
print(environment_yml)

--- requirements.txt ---
numpy==2.0.2
pandas==2.2.2
scikit-learn==1.6.1
matplotlib==3.10.0

--- environment.yml ---
name: py4fi-ch02
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.12
  - pip
  - pip:
      - numpy==2.0.2
      - pandas==2.2.2
      - scikit-learn==1.6.1
      - matplotlib==3.10.0



We have produced two files that, between them, are the **portable contract** for our environment. A colleague clones your repository, runs

```bash
conda env create -f environment.yml
conda activate py4fi-ch02
```

and gets a Python interpreter that is materially equivalent to ours. The `==` pins are strict, which is the right choice for backtests and paper replications. For library development you typically loosen them to `~=` (compatible release) or specify only a lower bound, because over-pinning makes your library uninstallable alongside other libraries.

**Lock files are the next level.** `requirements.txt` pins direct dependencies; a **lock file** (`uv.lock`, `poetry.lock`, `conda-lock`) pins direct *and transitive* dependencies, with content hashes. The difference matters: `requirements.txt` says "I want pandas 3.0.2"; a lock file says "I want pandas 3.0.2 *and* the specific NumPy 2.4.4 wheel with this SHA-256 *and* the exact CPython 3.12.3 binary." Lock files are how you turn "should reproduce" into "will reproduce."

For ML projects, the modern best practice as of 2026 is `pyproject.toml` for the spec and `uv.lock` for the lock. `uv` (written in Rust by the Astral team) installs roughly 10--50× faster than `pip` and produces lock files that work cross-platform. It is not yet what Hilpisch's book teaches, but it is what you should reach for first.

## 2.6 Virtual environments -- the conda way and the venv way

A virtual environment is, technically, **a directory with its own `python` binary, its own `site-packages`, and its own activation script** that prepends that directory to `PATH`. It is conceptually a sandbox: changes inside one virtual environment do not affect another, and removing one is `rm -rf` of a single directory.

The book teaches the conda approach. Here is the exact transcript of the textbook's session, reproduced as a reference shell block:

In [6]:
SHELL_BLOCK = r'''
# Textbook transcript — virtual environment with conda

# 1. Create a Python 2.7 environment alongside the default 3.7
$ conda create --name py27 python=2.7

# 2. Activate it (prompt now reads `(py27)` to show the active env)
$ conda activate py27

# 3. Install IPython into THIS environment only
(py27) $ conda install ipython

# 4. Confirm: this IPython runs Python 2.7 syntax
(py27) $ ipython
In [1]: print "Hello Python for Finance World!"     # 2.7-only syntax
Hello Python for Finance World!

# 5. List all environments on the machine
(py27) $ conda env list

# 6. Export the spec to a portable YAML file
(py27) $ conda env export --no-builds > py27env.yml

# 7. Recreate elsewhere
$ conda env create -f py27env.yml

# 8. Tear down when finished
$ conda deactivate
$ conda env remove --name py27
'''
print(SHELL_BLOCK)


# Textbook transcript — virtual environment with conda

# 1. Create a Python 2.7 environment alongside the default 3.7
$ conda create --name py27 python=2.7

# 2. Activate it (prompt now reads `(py27)` to show the active env)
$ conda activate py27

# 3. Install IPython into THIS environment only
(py27) $ conda install ipython

# 4. Confirm: this IPython runs Python 2.7 syntax
(py27) $ ipython
In [1]: print "Hello Python for Finance World!"     # 2.7-only syntax
Hello Python for Finance World!

# 5. List all environments on the machine
(py27) $ conda env list

# 6. Export the spec to a portable YAML file
(py27) $ conda env export --no-builds > py27env.yml

# 7. Recreate elsewhere
$ conda env create -f py27env.yml

# 8. Tear down when finished
$ conda deactivate
$ conda env remove --name py27



Two things to call out for the modern reader.

**Python 2.7 is dead.** The textbook's example uses 2.7 because in 2018 there was still a meaningful body of legacy code that needed it. Python 2.7 reached end-of-life on January 1, 2020. Today, the analogous example would be running a legacy 3.8 environment alongside a modern 3.12 environment -- still useful when a paper's code was written against an older PyTorch that no longer builds against current CUDA.

**`venv` is in the standard library and is enough for many cases.** If your project is pure Python (no CUDA, no exotic C extensions) you do not need conda; `python -m venv` plus `pip` is sufficient and lighter:

```bash
python3 -m venv .venv          # create
source .venv/bin/activate       # activate (Linux/macOS)
.venv\Scripts\activate.bat      # activate (Windows)
pip install -r requirements.txt
deactivate                      # leave
rm -rf .venv                    # destroy
```

For ML work, the rule of thumb in 2026:

- **`venv` + `uv`** for pure-Python or CPU-only projects.
- **`conda` (or `mamba`)** when you need CUDA, MKL, GDAL, or any other tricky binary dependency.
- **Docker** when the project also has system-level dependencies (Postgres client libs, particular compiler versions, OS libraries).

The choice is not religious. It is engineering: pick the lightest tool that solves your reproducibility problem.

## 2.7 Containers -- Docker, images, and the "class vs instance" analogy

The book introduces a piece of vocabulary that is genuinely useful even outside the Docker world:

> *"A Docker image is to a Docker container what a Python class is to an instance of that class."*

An **image** is the immutable, layered, content-addressed filesystem (effectively a frozen tarball of an OS plus your code). A **container** is a running process (or several) launched from that image, with its own writable upper layer that is discarded when the container stops. You can spin up 100 containers from one image; they are isolated from each other and from the host.

The textbook builds an Ubuntu-based image with Miniconda, pandas, and IPython preinstalled. The Dockerfile is:

In [7]:
SHELL_BLOCK = r'''
# Textbook Dockerfile (lightly annotated; reproduced from §2.7.2)

FROM ubuntu:latest                             # base image: latest Ubuntu
MAINTAINER yves
ADD install.sh /                               # copy the install script in
RUN chmod u+x /install.sh
RUN /install.sh                                # ... and execute it during build
ENV PATH /root/miniconda3/bin:$PATH            # so `python` resolves correctly
CMD ["ipython"]                                # default command when run

# install.sh (the heavy lifting):
#   apt-get update && apt-get upgrade -y
#   apt-get install -y bzip2 gcc git htop screen vim wget
#   wget https://repo.continuum.io/miniconda/Miniconda3-latest-Linux-x86_64.sh -O Miniconda.sh
#   bash Miniconda.sh -b
#   conda install -y pandas ipython
'''
print(SHELL_BLOCK)


# Textbook Dockerfile (lightly annotated; reproduced from §2.7.2)

FROM ubuntu:latest                             # base image: latest Ubuntu
MAINTAINER yves
ADD install.sh /                               # copy the install script in
RUN chmod u+x /install.sh
RUN /install.sh                                # ... and execute it during build
ENV PATH /root/miniconda3/bin:$PATH            # so `python` resolves correctly
CMD ["ipython"]                                # default command when run

# install.sh (the heavy lifting):
#   apt-get update && apt-get upgrade -y
#   apt-get install -y bzip2 gcc git htop screen vim wget
#   wget https://repo.continuum.io/miniconda/Miniconda3-latest-Linux-x86_64.sh -O Miniconda.sh
#   bash Miniconda.sh -b
#   conda install -y pandas ipython



This Dockerfile is functional but **not best-practice in 2026**. Three improvements every modern ML team makes:

1. **Pin the base image.** `FROM ubuntu:latest` means *the most recent ubuntu at the time the image is built* -- which can change between builds, breaking reproducibility. Use `FROM ubuntu:24.04` or, even better, `FROM ubuntu:24.04@sha256:<digest>`.
2. **Avoid rerunning installs in unrelated layers.** Docker caches layers by content. The textbook's `RUN /install.sh` bundles `apt-get update`, `wget Miniconda`, `conda install pandas`, *and* `pip install cufflinks` into a single layer that takes ~5 minutes to rebuild whenever any of them changes. Splitting the install into stable layers (system tools first, conda env second, pip extras third) lets cache reuse turn a 5-minute rebuild into a 5-second one.
3. **Pick a purpose-built ML base image.** The PyTorch team publishes `pytorch/pytorch:2.5.1-cuda12.4-cudnn9-runtime`, which is Ubuntu + CUDA + cuDNN + PyTorch already wired together. NVIDIA publishes the `nvidia/cuda` images. Hugging Face maintains TGI and Transformers images. **Inheriting from these saves you days of debugging CUDA compatibility issues.**

A modern equivalent of the textbook Dockerfile, suitable for a small ML/quant project, looks like this:

In [8]:
SHELL_BLOCK = r'''
# Modern data-science Dockerfile — best-practice as of 2026

# 1. Pinned base image (digest can be added with @sha256:...)
FROM python:3.12-slim-bookworm

# 2. System dependencies in one layer; --no-install-recommends keeps the image small
RUN apt-get update && \
    apt-get install -y --no-install-recommends \
        git curl build-essential ca-certificates && \
    rm -rf /var/lib/apt/lists/*

# 3. Install uv — the Rust-written package manager — via the official installer
ADD https://astral.sh/uv/install.sh /uv-install.sh
RUN sh /uv-install.sh && rm /uv-install.sh
ENV PATH="/root/.local/bin:${PATH}"

# 4. Copy ONLY the spec files first so this layer caches independently of source code
WORKDIR /app
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev

# 5. Copy source last — changes here do not invalidate the dependency layer
COPY . .

# 6. Non-root user for security
RUN useradd -m appuser && chown -R appuser /app
USER appuser

# 7. Explicit entrypoint
CMD ["uv", "run", "python", "-m", "myproject"]
'''
print(SHELL_BLOCK)


# Modern data-science Dockerfile — best-practice as of 2026

# 1. Pinned base image (digest can be added with @sha256:...)
FROM python:3.12-slim-bookworm

# 2. System dependencies in one layer; --no-install-recommends keeps the image small
RUN apt-get update && \
    apt-get install -y --no-install-recommends \
        git curl build-essential ca-certificates && \
    rm -rf /var/lib/apt/lists/*

# 3. Install uv — the Rust-written package manager — via the official installer
ADD https://astral.sh/uv/install.sh /uv-install.sh
RUN sh /uv-install.sh && rm /uv-install.sh
ENV PATH="/root/.local/bin:${PATH}"

# 4. Copy ONLY the spec files first so this layer caches independently of source code
WORKDIR /app
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev

# 5. Copy source last — changes here do not invalidate the dependency layer
COPY . .

# 6. Non-root user for security
RUN useradd -m appuser && chown -R appuser /app
USER appuser

# 7. Explicit entrypoint
CMD ["uv", "run", "pyt

The differences from the textbook Dockerfile are not aesthetic. They are operational:

- **Layer ordering**: dependency install separates from source code, so a code-only change rebuilds in seconds.
- **`uv sync --frozen`**: installs from the lock file with no network resolution. Reproducibility-by-default.
- **Non-root user**: the default Docker user is root, which means a container escape gives root on the host. ML systems shipped to customers must drop privileges.
- **No `MAINTAINER`**: that directive was deprecated in Docker 1.13 (2017). Use `LABEL maintainer="..."` instead.

The textbook then runs the container, lands in IPython, and types six lines that we will execute live below.

## 2.8 Reproducing the in-container example

Inside the Docker container, the textbook runs:

```python
In [1]: import numpy as np
In [2]: a = np.random.standard_normal((5, 3))
In [3]: import pandas as pd
In [4]: df = pd.DataFrame(a, columns=['a', 'b', 'c'])
In [5]: df
```

Note that the textbook does **not seed** the RNG before this call, so the array values are different in every run. We seed deliberately so the output below is reproducible.

In [9]:
import numpy as np
import pandas as pd

np.random.seed(2026)                       # deliberately seeded for reproducibility
a = np.random.standard_normal((5, 3))
df = pd.DataFrame(a, columns=['a', 'b', 'c'])
df

,a,b,c
0,-0.431719,-1.392874,0.311571
1,-0.013235,1.449708,0.298153
2,-0.829896,-1.596159,0.613438
3,-0.317597,-0.073997,-0.608631
4,-1.999873,0.908241,0.482980


The output is a $5 \times 3$ DataFrame with columns `a`, `b`, `c` of dtype `float64`. The values are drawn from $\mathcal{N}(0, 1)$ via the legacy seeded API and are bit-stable across NumPy versions for the reasons explained in §2.4.

We can perform the textbook's `df.info()` call to inspect the structure:

In [10]:
df.info()
print()
print('Memory footprint of the DataFrame values:')
print(f'  shape           : {df.shape}')
print(f'  dtype           : {df.dtypes.iloc[0]}')
print(f'  bytes per value : {df.values.itemsize}')
print(f'  total bytes     : {df.values.nbytes}')
print(f'  per-row mean    : {df.mean(axis=1).round(4).tolist()}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   a       5 non-null      float64
 1   b       5 non-null      float64
 2   c       5 non-null      float64
dtypes: float64(3)
memory usage: 252.0 bytes

Memory footprint of the DataFrame values:
  shape           : (5, 3)
  dtype           : float64
  bytes per value : 8
  total bytes     : 120
  per-row mean    : [-0.5043, 0.5782, -0.6042, -0.3334, -0.2029]


The DataFrame holds **5 rows × 3 columns = 15 values**, each a 64-bit (8-byte) float, for $5 \times 3 \times 8 = 120$ bytes of payload. pandas reports a slightly larger total in `info()` because the index, the column metadata, and the dtype object each carry overhead. The per-row mean ranges roughly $\pm 0.5$, which is consistent with three i.i.d. $\mathcal{N}(0, 1)$ samples having a sample mean with standard error $1 / \sqrt{3} \approx 0.58$.

**Why does the book bother with this trivial example?** Because the point is **not** the DataFrame -- it is that the *exact same six lines of Python ran inside an Ubuntu container with Miniconda installed*. The container abstraction lets you ship "Ubuntu + Python + pandas + your code" as one immutable artifact, push it to a registry, and pull it onto **any** Linux host -- your laptop, a staging server, a Kubernetes cluster, a Lambda function with container support. **The six lines of Python are reproducible because the entire OS underneath them is reproducible.**

In ML terms: this is how you ship a model. The container contains the inference code, the model weights, the exact PyTorch and CUDA versions, and the OS libraries the wheels link against. Without containers, deployment is "and then on production we discovered glibc 2.31 vs 2.35 changed the behavior of `strcoll`." With containers, deployment is `docker pull && docker run`.

## 2.9 Cloud instances -- DigitalOcean and the Jupyter-in-the-cloud pattern

The book's final infrastructure layer is the **cloud instance**: a virtual machine you rent by the hour. Hilpisch picks DigitalOcean for the demo because their 2018 pricing was 5 USD/month for the smallest "Droplet" (1 vCPU, 1 GB RAM, 25 GB SSD) -- about half the price of an equivalent AWS EC2 instance at the time.

The architectural goal is universal: **a browser-accessible Jupyter Notebook server**, password-protected and SSL-encrypted, running on cheap remote compute. Once that exists, you can develop from any device with a browser, and your data never leaves the server.

The full setup orchestration uses five files:

1. `setup.sh` -- runs locally, takes a Droplet IP, copies files via `scp`, runs `install.sh` over `ssh`.
2. `install.sh` -- runs on the Droplet, installs Miniconda, conda packages, and starts Jupyter.
3. `jupyter_notebook_config.py` -- Jupyter server configuration (port, password hash, SSL paths).
4. `cert.pem`, `cert.key` -- self-signed SSL certificate generated locally by `openssl req`.

The book reproduces the `openssl` invocation, the `install.sh` shell script, and the Jupyter config. We will do something more useful: **reproduce the password-hash generation step in pure Python**, and explain how Jupyter authenticates against it.

### 2.9.1 Reproducing `notebook.auth.passwd('jupyter')` in pure Python

The textbook generates a password hash in IPython:

```python
In [1]: from notebook.auth import passwd
In [2]: passwd('jupyter')
Out[2]: 'sha1:d4d34232ac3a:55ea0ffd78cc3299e3e5e6ecc0d36be0935d424b'
```

The format is `sha1:<hex_salt>:<sha1_hexdigest_of_password_plus_salt>`. Jupyter, on receiving a login attempt, parses this string, extracts the salt, computes `sha1(submitted_password + salt)`, and compares to the stored digest in constant time.

This is a 12-line algorithm; we reproduce it exactly in pure Python and verify that, given the textbook's salt, we get the textbook's hash:

In [11]:
import hashlib
import secrets


def passwd_legacy(password, salt=None):
    # Reproduce notebook.auth.passwd from the SHA-1 era (Jupyter < 5.0).
    # Format: 'sha1:<hex_salt>:<sha1(password + salt)_hexdigest>'
    if salt is None:
        salt = secrets.token_hex(6)              # 12 hex chars = 48 bits of entropy
    h = hashlib.sha1((password + salt).encode('utf-8')).hexdigest()
    return f'sha1:{salt}:{h}'


# Verify: with the SAME salt as the textbook, we MUST get the SAME hash.
book_salt = 'd4d34232ac3a'
book_hash = '55ea0ffd78cc3299e3e5e6ecc0d36be0935d424b'

ours = passwd_legacy('jupyter', salt=book_salt)
expected = f'sha1:{book_salt}:{book_hash}'
print(f'Our reproduction : {ours}')
print(f'Textbook reports : {expected}')
print(f'Match            : {ours == expected}')

# And what a fresh hash with a random salt looks like:
fresh = passwd_legacy('jupyter')
print(f'\nFresh hash       : {fresh}')

Our reproduction : sha1:d4d34232ac3a:55ea0ffd78cc3299e3e5e6ecc0d36be0935d424b
Textbook reports : sha1:d4d34232ac3a:55ea0ffd78cc3299e3e5e6ecc0d36be0935d424b
Match            : True

Fresh hash       : sha1:511cbdde4699:f906b1c453cd60aea2efa5e2b79ed72fed73968e


**The match is exact: bit-for-bit identical to what the textbook printed eight years ago.** This is not Jupyter being clever; it is that the algorithm is `sha1(password + salt)`, a deterministic function. Knowing the algorithm, we reimplement it in 12 lines and verify by reproducing the textbook's hash.

Two observations.

**SHA-1 is broken for cryptographic use.** In 2017, Google demonstrated a practical SHA-1 collision (the "SHAttered" attack). For a password verifier, collisions are less of an issue than **preimage resistance** plus **slowness against brute force** -- but SHA-1 fails on the second front: a single GPU computes billions of SHA-1 hashes per second. Modern Jupyter (since notebook 5.0, May 2017) defaults to **argon2id**, the password-hashing algorithm that won the 2015 Password Hashing Competition. argon2id is deliberately slow (configurable memory and time cost) and resists GPU attacks.

**You should never roll your own auth.** The reason we reproduced `passwd_legacy` here was pedagogical -- to show that the format is just `sha1(pw + salt)`. In production, even for a quick Jupyter server, **never** reuse this code. Use `argon2-cffi`'s `PasswordHasher`, or better yet, put the Jupyter server behind an authenticated proxy (Caddy with Tailscale, Cloudflare Access, AWS IAM-authenticated ALB) and skip Jupyter's auth entirely.

The textbook's full Jupyter config file references the hash in `c.NotebookApp.password = 'sha1:...'`, the SSL certificate in `c.NotebookApp.certfile`, and binds to port 8888. We omit reproducing that file here; on a modern install you would write a `jupyter_lab_config.py` instead, and most people use **JupyterHub** for any deployment with more than one user.

### 2.9.2 What replaces the Droplet + Jupyter pattern in 2026

The architecture the book describes -- "rent a small VM, install Jupyter, expose it on the internet, secure with SSL" -- is still completely valid. It is also **no longer the obvious default**, because cheaper and easier alternatives have appeared. A short field guide for an ML practitioner today:

- **Google Colab** (free tier with T4 GPUs; Pro tier with A100 access). Zero infrastructure to manage. Best for prototyping, paper replication, and teaching.
- **Hugging Face Spaces** (free CPU; paid GPU). Best for shipping a model behind a public Gradio or Streamlit UI.
- **Modal, Lambda Labs, Runpod, Vast.ai**. Per-second GPU billing, container-native. Best for training and inference of larger models. Replaced "rent an EC2 instance for a month" for most academic ML by 2024.
- **AWS SageMaker, Azure ML, GCP Vertex AI**. Enterprise-grade, integrated with auth, audit, and data lakes. Best for regulated industries and corporate teams.
- **GitHub Codespaces, Gitpod, Coder**. Browser-based development environments backed by containers. Replaced "ssh to a Droplet running Jupyter" for many engineers.
- **Self-hosted JupyterHub on Kubernetes**. Best for university and research-lab settings with many users sharing GPU resources.

Hilpisch's pattern -- DigitalOcean + self-installed Jupyter + self-signed SSL -- is the **maximally educational** version, because you build every piece yourself. We do not run it here, but understanding what each command does is what lets you reach for the right managed alternative when you need one.

## 2.10 The reproducibility hierarchy -- one diagram to remember

The four tools the chapter covers form a **strict hierarchy**, where each layer reproduces more of the runtime than the one below. This is the single most important diagram an ML engineer should internalize:

In [12]:
# A textual rendering of the reproducibility hierarchy.
# Each layer subsumes everything below it.
diagram = [
    "+---------------------------------------------------------------+",
    "|  Level 4: CLOUD INSTANCE                                      |",
    "|    Reproduces: hardware, network, OS, runtime, deps, code     |",
    "|    Tools:      AWS, GCP, Azure, DigitalOcean, Modal, Lambda   |",
    "|    When:       multi-user, regulated, GPU-bound, prod         |",
    "+---------------------------------------------------------------+",
    "                            ^ subsumes                          ",
    "+---------------------------------------------------------------+",
    "|  Level 3: CONTAINER (Docker / OCI image)                      |",
    "|    Reproduces: OS, runtime, deps, code                        |",
    "|    Tools:      Docker, podman, Kaniko, Buildah                |",
    "|    When:       cross-machine deployment, microservices, ML    |",
    "|                inference, anything with a system-lib dep      |",
    "+---------------------------------------------------------------+",
    "                            ^ subsumes                          ",
    "+---------------------------------------------------------------+",
    "|  Level 2: VIRTUAL ENVIRONMENT                                 |",
    "|    Reproduces: Python interpreter version, packages           |",
    "|    Tools:      venv, virtualenv, conda, mamba, uv, poetry     |",
    "|    When:       single-machine dev, multi-project laptops      |",
    "+---------------------------------------------------------------+",
    "                            ^ subsumes                          ",
    "+---------------------------------------------------------------+",
    "|  Level 1: PINNED PACKAGE LIST                                 |",
    "|    Reproduces: package versions (assuming compatible Python)  |",
    "|    Tools:      requirements.txt, pyproject.toml, uv.lock      |",
    "|    When:       libraries, simple scripts, CI                  |",
    "+---------------------------------------------------------------+",
    "                            ^ subsumes                          ",
    "+---------------------------------------------------------------+",
    "|  Level 0: SEEDED RANDOMNESS                                   |",
    "|    Reproduces: stochastic algorithm outputs                   |",
    "|    Tools:      np.random.seed, torch.manual_seed, env vars    |",
    "|    When:       always -- every notebook, every experiment     |",
    "+---------------------------------------------------------------+",
]
print('\n'.join(diagram))

+---------------------------------------------------------------+
|  Level 4: CLOUD INSTANCE                                      |
|    Reproduces: hardware, network, OS, runtime, deps, code     |
|    Tools:      AWS, GCP, Azure, DigitalOcean, Modal, Lambda   |
|    When:       multi-user, regulated, GPU-bound, prod         |
+---------------------------------------------------------------+
                            ^ subsumes                          
+---------------------------------------------------------------+
|  Level 3: CONTAINER (Docker / OCI image)                      |
|    Reproduces: OS, runtime, deps, code                        |
|    Tools:      Docker, podman, Kaniko, Buildah                |
|    When:       cross-machine deployment, microservices, ML    |
|                inference, anything with a system-lib dep      |
+---------------------------------------------------------------+
                            ^ subsumes                          
+-----------

**The rule of thumb**: pick the lowest level on the hierarchy that solves your problem. Over-engineering reproducibility (Kubernetes for a one-off paper) is its own form of waste; under-engineering it (no seed, no pin) is technical debt that compounds.

For a typical ML/quant project in 2026:

- **Level 0** is non-negotiable. Every experiment seeds.
- **Level 1** (`pyproject.toml` + lock file) is also non-negotiable for any code that lives longer than a week.
- **Level 2** (`venv` or `conda` env) is the daily-driver workspace boundary.
- **Level 3** (Docker) appears when you ship -- to a teammate, to a training cluster, to a customer.
- **Level 4** (cloud) is where training and inference actually happen.

The textbook covers all five levels (counting seeded randomness as Level 0). Subsequent chapters of the book build on top of this stack -- the financial analytics in Chapters 3--12 assume you have a working Python install, the algorithmic-trading code in Chapters 14--16 assumes you can reach a broker API from a cloud machine, and the derivatives library in Chapters 17--21 assumes you can pin numerical libraries that produce identical pricing to the last decimal across re-runs.

## 2.11 Conclusion

Python's deployment story has improved dramatically since this book's first edition (2014) and meaningfully since the second (2018). The four-layer hierarchy -- pin, env, container, cloud -- is mature, and 2026's tooling (`uv`, `mamba`, `podman`, container-native cloud platforms) makes each layer faster and safer than the equivalents the book describes.

For an ML/quant practitioner the core lessons are unchanged from what Hilpisch wrote eight years ago:

- **Always pin.** A backtest that does not pin its dependencies is a hypothesis, not a result.
- **Always seed.** A model that does not set every relevant RNG seed is a non-result.
- **Use environments as workspaces, not garbage dumps.** A virtual environment per project, recreated cheaply from spec.
- **Containers when you ship, not before.** The right time to introduce Docker is the moment your code first needs to leave your laptop.
- **Cloud when scale or availability requires it.** The smallest Droplet at 5 USD/month is still a fine teaching environment; a multi-A100 node at 30 USD/hour is what you reach for when training a 7B model. Pick the smallest box that does the job, and turn it off when you are done.

Subsequent chapters use the infrastructure built here. Chapter 3 introduces NumPy in earnest; Chapter 5 turns to pandas and time series; the algorithmic-trading chapters in Part IV assume the cloud-based Jupyter setup outlined in §2.9 is available; the derivatives library in Part V assumes you can install scientific stacks reliably on any machine you rent.

> *Sharpen your tools first, then go build the house.* Musashi was talking about wood; we are talking about Python. The lesson is the same.